<h3>실습 안내</h3>
<p>YAML 설정을 읽고 환경 변수로 검색 개수를 바꿉니다. 제공된 18개 PDF와 Ch 11의 Q001을 사용해 검색과 답변 생성을 실행한 뒤, 설정·순위가 있는 검색 결과·답을 JSONL 한 행으로 저장하고 다시 읽습니다.</p>

In [1]:
# YAML·검색·생성 실습에 필요한 패키지를 현재 커널에 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3" "pyyaml==6.0.3" "pypdf==6.17.0"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키와 실습 경로 준비</h3><p><code>.env</code>에서 키의 존재만 확인합니다. 압축을 풀어 실행해도 제공 PDF, Ch 11 질문셋, 챕터 로그 폴더를 찾도록 실습 최상위 경로를 확인합니다.</p>

In [2]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
root = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "labs/data/rag-corpus").exists())
chapter_dir = root / "labs/part3/ch09-rag-first-run"
runs_dir = chapter_dir / "runs"
temp_dir = TemporaryDirectory()
work_dir = Path(temp_dir.name)
print("API 키 준비 완료")
print("임시 실험 폴더:", work_dir.name)

API 키 준비 완료
임시 실험 폴더: tmpg294gm1q


<h3>2. YAML 설정 파일 만들기</h3><p>바꿀 값을 코드 여러 곳에 흩어 놓지 않고 YAML 한 곳에 모읍니다.</p>

In [3]:
config_path = work_dir / "config.yaml"
config_path.write_text("""name: baseline
split:
  chunk_size: 500
  chunk_overlap: 50
embedding:
  model: text-embedding-3-small
retrieval:
  k: 1
generation:
  model: openai:gpt-5.4-mini
qid: Q001
gold_doc_ids: [E01]
question: 미르시립도서관의 정기 휴관일은 언제인가요?
""", encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))

name: baseline
split:
  chunk_size: 500
  chunk_overlap: 50
embedding:
  model: text-embedding-3-small
retrieval:
  k: 1
generation:
  model: openai:gpt-5.4-mini
qid: Q001
gold_doc_ids: [E01]
question: 미르시립도서관의 정기 휴관일은 언제인가요?



<h3>3. YAML을 읽고 실행 조건 하나 바꾸기</h3><p><code>RETRIEVAL_K</code> 환경 변수가 있으면 그 값을 사용하고, 없으면 2를 사용합니다. 기본값 1과 변경값을 비교합니다.</p>

In [4]:
from copy import deepcopy

import yaml

config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
run_config = deepcopy(config)
run_config["name"] = os.getenv("RUN_NAME", "k2")
run_config["retrieval"]["k"] = int(os.getenv("RETRIEVAL_K", "2"))
print("기본 k:", config["retrieval"]["k"])
print("실행 k:", run_config["retrieval"]["k"])
assert config["split"] == run_config["split"]
assert config["embedding"] == run_config["embedding"]

기본 k: 1
실행 k: 2


<h3>4. 색인 설정을 구분하는 해시 만들기</h3><p>문서 색인에 영향을 주는 분할·임베딩 설정만 정렬된 JSON 문자열로 바꾼 뒤 SHA-256 해시를 계산합니다.</p>

In [5]:
import hashlib
import json

index_settings = {
    "split": run_config["split"],
    "embedding": run_config["embedding"],
}
index_json = json.dumps(index_settings, sort_keys=True)
config_sha = hashlib.sha256(index_json.encode()).hexdigest()[:10]
corpus_sha = hashlib.sha256((root / "labs/data/rag-corpus/manifest.json").read_bytes()).hexdigest()
dataset_sha = hashlib.sha256((root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl").read_bytes()).hexdigest()
print("색인 설정:", index_json)
print("설정 해시:", config_sha)

색인 설정: {"embedding": {"model": "text-embedding-3-small"}, "split": {"chunk_overlap": 50, "chunk_size": 500}}
설정 해시: d853036bf1


<h3>5. 제공 PDF의 실제 임베딩 색인과 검색 실행</h3><p>Ch 11이 평가하는 것과 같은 18개 디지털 PDF를 쪽별 <code>Document</code>로 만듭니다. 청크에 문서 ID·쪽·청크 ID를 남기고 Q001을 검색합니다.</p>

In [6]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

pdf_paths = sorted(path for tier in ("easy", "medium", "hard") for path in (root / "labs/data/rag-corpus/pdf" / tier).glob("*.pdf"))
documents = []
for path in pdf_paths:
    for page, pdf_page in enumerate(PdfReader(path).pages, start=1):
        documents.append(Document(page_content=pdf_page.extract_text(), metadata={"doc_id": path.stem, "page": page, "source": path.name}))
splitter = RecursiveCharacterTextSplitter(**run_config["split"])
chunks = splitter.split_documents(documents)
for number, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"ch09:{config_sha}:{number:04d}"
embedding = OpenAIEmbeddings(model=run_config["embedding"]["model"])
store = InMemoryVectorStore.from_documents(chunks, embedding)
hits = store.similarity_search_with_score(run_config["question"], k=run_config["retrieval"]["k"])
print("디지털 PDF:", len(pdf_paths), "쪽:", len(documents), "청크:", len(chunks))
print("검색 ID:", [hit.metadata["doc_id"] for hit, _ in hits])
print("첫 근거:", hits[0][0].page_content[:120])

디지털 PDF: 18 쪽: 93 청크: 176
검색 ID: ['E01', 'E01']
첫 근거: 미르시립도서관 이용 안내
미르시립도서관
2026년판
발행 2026. 3. 1.
1


<h3>6. 검색 근거로 답변 생성</h3><p>검색한 본문과 출처 ID를 프롬프트에 넣고 실제 채팅 모델을 호출합니다.</p>

In [7]:
from langchain.chat_models import init_chat_model

context = "\n".join(
    f"[{hit.metadata['doc_id']} p.{hit.metadata['page']}] {hit.page_content}" for hit, _ in hits
)
prompt = f"""다음 근거만 사용해 질문에 답하세요.
근거에 답이 없으면 '확인할 수 없습니다.'라고 답하세요.

근거:
{context}

질문: {run_config['question']}
답에 사용한 출처를 [문서ID p.쪽] 형식으로 표시하세요."""
model = init_chat_model(run_config["generation"]["model"])
response = model.invoke(prompt)
answer = response.text
print(answer)

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]


<h3>7. Ch 11이 읽을 JSONL 저장하고 다시 읽기</h3><p>질문 ID·정답 문서 ID·순위가 있는 검색 결과·설정·생성 답을 한 행에 저장합니다. 실행 ID별 파일은 보존하고, 고정 이름 파일은 다음 챕터가 읽는 최신 포인터입니다.</p>

In [8]:
from datetime import UTC, datetime
source_run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f-ch09")
retrieved = [{"doc_id": hit.metadata["doc_id"], "chunk_id": hit.metadata["chunk_id"], "page": hit.metadata["page"], "rank": rank, "score": round(score, 6)} for rank, (hit, score) in enumerate(hits, start=1)]
logged_config = {key: run_config[key] for key in ("name", "split", "embedding", "retrieval", "generation")} | {"inputs": {"corpus_scope": "18 digital PDFs", "corpus_manifest_sha256": corpus_sha, "evaluation_dataset_sha256": dataset_sha}}
record = {
    "qid": run_config["qid"],
    "question": run_config["question"],
    "gold_doc_ids": run_config["gold_doc_ids"],
    "source_chapter": "ch09",
    "source_run_id": source_run_id,
    "config": logged_config,
    "retrieved": retrieved,
    "answer": answer,
}
line = json.dumps(record, ensure_ascii=False) + "\n"
archive = runs_dir / "continuity" / f"{source_run_id}.jsonl"
archive.parent.mkdir(parents=True, exist_ok=True)
archive.write_text(line, encoding="utf-8")
pointer = runs_dir / "continuity-ch09-search.jsonl"
pointer.with_suffix(".tmp").write_text(line, encoding="utf-8")
pointer.with_suffix(".tmp").replace(pointer)
saved = json.loads(pointer.read_text(encoding="utf-8"))
print("보존/다음 챕터 로그:", archive.relative_to(root), pointer.relative_to(root))
print(saved["qid"], saved["gold_doc_ids"], saved["retrieved"][0])
assert saved == record

보존/다음 챕터 로그: labs/part3/ch09-rag-first-run/runs/continuity/20260909T161218875711-ch09.jsonl labs/part3/ch09-rag-first-run/runs/continuity-ch09-search.jsonl
Q001 ['E01'] {'doc_id': 'E01', 'chunk_id': 'ch09:d853036bf1:0000', 'page': 1, 'rank': 1, 'score': 0.649152}


<h3>8. 조건을 붙여 명령 하나로 다시 실행하기</h3><p>압축을 풀고 <code>part3-practice</code> 폴더에서 아래 명령을 실행하면 Jupyter가 새 커널로 노트북 전체를 순서대로 실행합니다.</p>

In [9]:
command = (
    "RUN_NAME=k4 RETRIEVAL_K=4 uv run --project labs/part3/ch09-rag-first-run "
    "--with nbconvert==7.17.0 --with ipykernel==7.3.0 --with pip "
    "jupyter nbconvert --to notebook --execute --inplace "
    "labs/part3/ch09-rag-first-run/notebooks/06-실험-하네스.ipynb --ExecutePreprocessor.timeout=240"
)
print(command)

RUN_NAME=k4 RETRIEVAL_K=4 uv run --project labs/part3/ch09-rag-first-run --with nbconvert==7.17.0 --with ipykernel==7.3.0 --with pip jupyter nbconvert --to notebook --execute --inplace labs/part3/ch09-rag-first-run/notebooks/06-실험-하네스.ipynb --ExecutePreprocessor.timeout=240


<h3>정리</h3><p>YAML을 읽고 조건 하나를 바꾼 뒤 검색과 생성을 실행했습니다. 실행 설정과 결과를 JSONL 한 행에 저장하고, 같은 노트북을 조건과 함께 실행하는 명령도 확인했습니다.</p>

<p><strong>원인</strong> — 청크 크기를 바꾸면 색인이 달라지지만, 이전 설정 해시를 그대로 사용하면 잘못된 색인을 선택합니다.</p>

In [10]:
from copy import deepcopy

failure_documents = [documents[1]]
baseline_chunks = splitter.split_documents(failure_documents)
baseline_store = InMemoryVectorStore.from_documents(baseline_chunks, embedding)
changed_split = deepcopy(run_config)
changed_split["split"]["chunk_size"] = 100
wrong_hits = baseline_store.similarity_search(run_config["question"], k=1)
new_settings = {
    "split": changed_split["split"],
    "embedding": changed_split["embedding"],
}
new_json = json.dumps(new_settings, sort_keys=True)
new_sha = hashlib.sha256(new_json.encode()).hexdigest()[:10]
print("이전 청크:", wrong_hits[0].page_content)
print("이전 해시:", config_sha, "변경 해시:", new_sha)
try:
    assert new_sha == config_sha
except AssertionError:
    print("예상한 실패: 변경된 설정에는 새 색인이 필요합니다.")

이전 청크: 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로"입니다. 시민 모두가 자료를 자유롭게 이용하고 배울 수 있도
록 돕는 것을 최우선으로 합니다.
2. 이용 시간
평일(화~금) 이용 시간은 오전 9시부터 오후 10시까지입니다. 토요일과 일요일은 오전 9시부터 오후 6
시까지 문을 엽니다.
매주 월요일은 정기 휴관일 입니다. 법정 공휴일에도 휴관하며, 공휴일이 월요일과 겹치면 그 다음 날인
화요일에도 휴관합니다.
어린이자료실(1층)은 요일과 관계없이 오후 6시에 문을 닫습니다. 디지털자료실(2층)은 오후 9시까지
이용할 수 있습니다.
구역 평일 주말 비고
종합자료실(2층) 09:00~22:00 09:00~18:00
이전 해시: d853036bf1 변경 해시: 142b5e9fe7
예상한 실패: 변경된 설정에는 새 색인이 필요합니다.


<p><strong>수정</strong> — 변경된 설정으로 문서를 다시 나누고 새 임베딩 색인을 만듭니다.</p>

In [11]:
new_splitter = RecursiveCharacterTextSplitter(**changed_split["split"])
new_chunks = new_splitter.split_documents(failure_documents)
new_store = InMemoryVectorStore.from_documents(new_chunks, embedding)
new_hits = new_store.similarity_search(run_config["question"], k=1)
print("변경 청크 수:", len(new_chunks))
print("변경 첫 청크:", new_hits[0].page_content)
print("변경 첫 청크 길이:", len(new_hits[0].page_content))
assert new_sha != config_sha
assert len(new_hits[0].page_content) <= 100

변경 청크 수: 15
변경 첫 청크: 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
변경 첫 청크 길이: 68
